# Post process data extracted using LLMs
1. Load data
2. Convert to correct format (numeric for numerical columns)
3. Eventually explode lists for geocoding?
4. Geocoding
5. Sanity checks

In [20]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
import regex as re
from matplotlib import pyplot as plt
from src.data import *
from src.plot_functions import *
from src.post_process_functions import *
from src.geocoding import *
from src.hazard_def import *
from src.impact_def import *
from src.sanity_checks import *



In [21]:
#load data (model)
model_name = "meta-llama/llama-4-scout-17b-16e-instruct"
nreports = 50
res_savename = f"llm_response_impact_labelled_reports_{model_name.replace('/', '_')}.csv"
response_df = pd.read_csv(DATA_OUT_LLMS+res_savename)

#load data (labelled)
#res_savename = "labelled_reports_impacts_all.csv"
#response_df = pd.read_csv(DATA_LABELLED+res_savename)


In [22]:
#load data (labelled)
#fnla = "labelled_8reports_impact_laura.csv"
#labelled_laura = pd.read_csv(DATA_LABELLED+fnla)
#fnlu = "labelled_reports_impacts_luca.csv"
#labelled_luca = pd.read_csv(DATA_LABELLED+fnlu)#.drop(["Unnamed: 0"],axis=1)
#
##reformat to be consistent
#labelled_laura["reportDate"] = pd.to_datetime(labelled_laura["reportDate"], dayfirst=True) #reformat date to be consistent
#labelled_luca["reportDate"] = pd.to_datetime(labelled_luca["reportDate"]) #reformat date to be consistent
#labelled_luca.rename({"annotation":"impactsAnnotation", "impactSubType":"impactSubtype"},inplace=True, axis=1)
#labelled_laura.rename({"annotation":"impactsAnnotation", "impactSubType":"impactSubtype"},inplace=True, axis=1)
#
#response_df = pd.concat([labelled_laura, labelled_luca]).reset_index(drop=True)
#res_savename = "labelled_reports_impacts_all.csv"
#response_df.to_csv(DATA_LABELLED+res_savename, index=False)


In [23]:
response_df

,impactSubtype,impactValue,impactUnit,impactValuePrecision,country,location,startYear,startMonth,startDay,endYear,...,impactsAnnotation,impactValueMin,impactValueMax,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text,impactType
0,Affected People,43880.0,people,exact,"['Kiribati', 'Papua New Guinea', 'Solomon Isla...",NaN,NaN,NaN,NaN,NaN,...,['TC PAM Global Relief Response : Across5 coun...,43880.0,43880.0,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN
1,Affected People,34573.0,people,exact,"['Kiribati', 'Papua New Guinea', 'Solomon Isla...",NaN,NaN,NaN,NaN,NaN,...,['TC PAM Global Recovery Response : Across5 co...,34573.0,34573.0,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN
2,Residential Buildings,900.0,houses,exact,['Vanuatu'],['West Tanna'],NaN,NaN,NaN,NaN,...,['This is important feedback as the safe shelt...,900.0,900.0,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN
3,Human Health and Wellbeing,85.0,per cent,exact,['Vanuatu'],NaN,NaN,NaN,NaN,NaN,...,"['In Vanuatu, the results of feedback mechanis...",85.0,85.0,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN
4,"Access to Water, Sanitation, and Hygiene",138.0,households,exact,['Tuvalu'],NaN,NaN,NaN,NaN,NaN,...,['A total of13 rainwater harvesting systems ( ...,138.0,138.0,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272,Crop Production and Forestry,NaN,NaN,NaN,['Zambia'],NaN,2023.0,NaN,NaN,2024.0,...,"['projected production levels were minimal, an...",NaN,NaN,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN
273,Affected Livestock and Animals,NaN,NaN,NaN,['Zambia'],NaN,2023.0,NaN,NaN,2024.0,...,['almost half of surveyed households that kept...,NaN,NaN,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN
274,Other Economic and Livelihood Impacts,11.0,CHF million,exact,['Zambia'],NaN,2024.0,NaN,NaN,NaN,...,"['the IFRC, in support to the ZRCS, launched a...",11.0,11.0,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN
275,Access to Food,NaN,NaN,NaN,['Zambia'],NaN,NaN,NaN,NaN,NaN,...,['decreased access to water has also led to ou...,NaN,NaN,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN


In [24]:
#get rid of nans
response_df = response_df.dropna(subset=["nathaz_text"]) if "nathaz_text" in response_df.columns else response_df

In [25]:
#convert numerical columns
num_cols = ["impactValue"]#"startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"
list_cols = ["location", "hazards", "impactsAnnotation"]
response_df_proc = cp.deepcopy(response_df)
response_df_proc = format_output(response_df_proc, num_cols=num_cols, list_cols=list_cols)



In [26]:
#add iso3
response_df_proc["country_iso3"] = response_df_proc["country"].apply(country_name_to_iso3)
response_df_proc["country_iso3_kw"] = response_df_proc["country_kw"].apply(country_name_to_iso3) if "country_kw" in response_df_proc.columns else None

In [27]:
#format units
from spacy.lang.en import English
from spacy.lang.punctuation import TOKENIZER_PREFIXES, TOKENIZER_SUFFIXES, TOKENIZER_INFIXES
from spacy.lang.en import TOKENIZER_EXCEPTIONS
from spacy.tokenizer import Tokenizer
from spacy.util import compile_prefix_regex, compile_suffix_regex, compile_infix_regex

## Post process
1. Reclassify hazards
2. Reclassify impactSubtypes
3. Reclassify units

In [28]:
impactSubtype_list

['Affected People',
 'Injured People',
 'Displaced People',
 'Homeless People',
 'Missing People',
 'Human Deaths',
 'Human Health and Wellbeing',
 'Infected and Ill People',
 'Road Infrastructure',
 'Other Transportation Infrastructure',
 'Water, Sanitation, and Hygiene Infrastructure',
 'Healthcare Infrastructure',
 'IT and Communication Infrastructure',
 'Residential Buildings',
 'Informal settlements',
 'Education Infrastructure',
 'Power and Energy Production Infrastructure',
 'Agriculture Infrastructure',
 'Crop Production and Forestry',
 'Affected Livestock and Animals',
 'Other Economic and Livelihood Impacts',
 'Recreation, Tourism, and Culture',
 'Access to Healthcare',
 'Access to transport and Mobility',
 'Water Quality and Availability',
 'Access to Education',
 'Access to Power and Energy',
 'Access to Food',
 'Access to Water, Sanitation, and Hygiene',
 'Other Human Impacts',
 'Other Infrastructure Impacts',
 'Other Agricultural Impacts',
 'Other Service Access Impacts']

In [29]:
#reclassify impacType
impact_kw_reclass = {
    'Crop Production and Forestry' : r"agri.*|forest",
    'Affected Animals' : r"livestock.*|animal.*|fish.*|cattle.*",
    'Other Infrastructure Impacts' : r"Other Infrastructur.* Impacts"
}
def reclassify_impact_subtype(extracted_data, allowed_impact_types, impact_kw_reclass):
    def reclass_impact_subtype(x):
        if x["impactSubtype"] in allowed_impact_types:
            return x["impactSubtype"]
        candidates = []
        for key, value in impact_kw_reclass.items():
            if re.search(value, x["impactSubtype"], re.IGNORECASE):
                candidates.append(key)
        if len(candidates) == 1:
            return candidates[0]
        else:
            return "Unknown"
    extracted_data["impactSubtype"] = extracted_data.apply(reclass_impact_subtype, axis=1)
    return extracted_data

response_df_proc = reclassify_impact_subtype(response_df_proc, impactSubtype_list, impact_kw_reclass)


In [ ]:
#reclassify hazard
hazard_kw_reclass ={
                        'Drought': r"drought.*|dry spell.*",
                        'Wildfire': r"fire.*|forestfire.*|wildfire.*|landfire.*|bushfire.*|forest fire.*|wild fire.*|land fire.*|bush fire.*" ,
                        'Earthquake' : r"earthquake.*|ground movement.|tsunami.",
                        'Mass movement': r"mass movement.*|avalanche.|land slide.*|landslide.*|rockfall.*|sudden subsidence.|mudslide.|mass movement.*",
                        'Volcanic activity' : r"volcanic.*|ash fall.|lava flow.|pyroclastic flow.|lahar",
                        'Flood': r"flood.*|inundation.*|coastal flood.|flash flood.|riverine flood.|ice jam flood.",
                        'Wave action' : r"|wave.*|rogue wave.|seiche",
                        'Extreme cold temperature' : r"extreme cold temperature.*|cold wave.*|coldwave.*|cold spell.*|severe winter conditions.",
                        'Extreme warm temperature' : r"extreme warm temperature.*|heat wave.*|heatwave.*|heat episode.*|((heat|hot) spell).*|heat stress.",
                        'Tropical storm' : r"tropical storms?|tropical storms?|typhoons?|hurricanes?|cyclonic storms?",
                        'Other storm' : r"extra-tropical storms?|winterstorms?|storms? surges?|winter storms?|extra tropical storms?|superstorms?|windstorms?|snowstorms?|blizzards?|convective storms?|derechos?|hail.*|lightning.*|tornado.*|thunderstorms?",
                        "Epidemics" : r"cholera|dengue|outbreak|epidemic|epidemic.*",
                        "Conflict" : r"conflict.*|war.*|terrorism.*|unrest.*",
                         }

def reclassify_hazard(extracted_data, hazard_kw_reclass):
    def reclass_haz(x):
        corr_haz = cp.deepcopy(x["hazards"])
        if any([haz for haz in x["hazards"] if haz not in hazard_kw_reclass.keys()]):
            for i, haz in enumerate(x["hazards"]):
                if haz not in hazard_kw_reclass.keys():
                    candidates = [haz_corr for haz_corr in hazard_kw_reclass.keys() if re.search(hazard_kw_reclass[haz_corr], haz, re.IGNORECASE)]
                    if len(candidates) == 1:
                        corr_haz[i] = candidates[0]
                    else:
                        corr_haz[i] = "Unknown"
        return corr_haz
    extracted_data["hazards_reclass"] = extracted_data.apply(reclass_haz, axis=1)
    return extracted_data

response_df_proc = reclassify_hazard(response_df_proc, hazard_kw_reclass)
explode_lists(response_df_proc).hazards_reclass.value_counts()

/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:89: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  repeat_counts = df[list_columns].applymap(len).max(axis=1)


hazards_reclass
Flood                       179
Unknown                     168
Extreme cold temperature     80
Wildfire                     72
Drought                      41
Volcanic activity            25
Mass movement                15
Earthquake                   14
Wave action                  14
Name: count, dtype: int64

In [31]:
wrong_haz = explode_lists(response_df_proc).copy()
wrong_haz = wrong_haz[wrong_haz["hazards_reclass"] == "Unknown"]
wrong_haz

/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:89: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  repeat_counts = df[list_columns].applymap(len).max(axis=1)


,impactSubtype,impactValue,impactUnit,impactValuePrecision,country,location,startYear,startMonth,startDay,endYear,...,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text,impactType,country_iso3,country_iso3_kw,hazards_reclass
0,Affected People,43880.0,people,exact,"['Kiribati', 'Papua New Guinea', 'Solomon Isla...",NaN,NaN,NaN,NaN,NaN,...,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,None,FJI,Unknown
1,Affected People,34573.0,people,exact,"['Kiribati', 'Papua New Guinea', 'Solomon Isla...",NaN,NaN,NaN,NaN,NaN,...,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,None,FJI,Unknown
2,Residential Buildings,900.0,houses,exact,['Vanuatu'],West Tanna,NaN,NaN,NaN,NaN,...,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,None,FJI,Unknown
3,Human Health and Wellbeing,85.0,per cent,exact,['Vanuatu'],NaN,NaN,NaN,NaN,NaN,...,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,None,FJI,Unknown
4,"Access to Water, Sanitation, and Hygiene",138.0,households,exact,['Tuvalu'],NaN,NaN,NaN,NaN,NaN,...,MDR55001,Fiji,2017-03-31 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,None,FJI,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
591,Other Infrastructure Impacts,NaN,NaN,NaN,['Yemen'],Nuristan,2022.0,5,NaN,2022.0,...,MDRYE011,Yemen,2023-05-29 00:00:00,https://go-api.ifrc.org/api/DownloadFile/67143...,Flood,"[""SITUATION ANALYSIS Description of the disast...",NaN,None,YEM,Unknown
601,Infected and Ill People,21000.0,people,exact,['Zambia'],Farah,2023.0,10,NaN,NaN,...,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN,None,ZMB,Unknown
602,Human Health and Wellbeing,NaN,NaN,NaN,['Zambia'],Faryab,NaN,NaN,NaN,NaN,...,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN,None,ZMB,Unknown
604,Affected Livestock and Animals,NaN,NaN,NaN,['Zambia'],Herat,2023.0,NaN,NaN,2024.0,...,MDRZM022,Zambia,2024-10-17 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN,None,ZMB,Unknown


In [32]:
impactSubtype_list

['Affected People',
 'Injured People',
 'Displaced People',
 'Homeless People',
 'Missing People',
 'Human Deaths',
 'Human Health and Wellbeing',
 'Infected and Ill People',
 'Road Infrastructure',
 'Other Transportation Infrastructure',
 'Water, Sanitation, and Hygiene Infrastructure',
 'Healthcare Infrastructure',
 'IT and Communication Infrastructure',
 'Residential Buildings',
 'Informal settlements',
 'Education Infrastructure',
 'Power and Energy Production Infrastructure',
 'Agriculture Infrastructure',
 'Crop Production and Forestry',
 'Affected Livestock and Animals',
 'Other Economic and Livelihood Impacts',
 'Recreation, Tourism, and Culture',
 'Access to Healthcare',
 'Access to transport and Mobility',
 'Water Quality and Availability',
 'Access to Education',
 'Access to Power and Energy',
 'Access to Food',
 'Access to Water, Sanitation, and Hygiene',
 'Other Human Impacts',
 'Other Infrastructure Impacts',
 'Other Agricultural Impacts',
 'Other Service Access Impacts']

### Units reclassification
1. Standardize units i.e. metric units to SI or common units.
2. Determine unit typology (e.g. distance, surface, weight, percent)
3. Convert non-metric units e.g. households to people, USD to CHF
4. Standardize non-metric units i.e. person, children to people, hospitals to health facilities

TO DOS
1. Handle deaths so that they dont go in other categories
2. Handle targeted people

In [33]:
#reclassify units
unit_converter = {"families" : (3, "people"),
                  "households": (3, "people"),
                  "village": (1000, "people"),
                  "communities": (100, "people"),
                  "USD": (1, "CHF"),
                  "$" : (1, "CHF"),
                  "EUR": (1, "CHF"),
                  "€": (1, "CHF"),
                  }

unit_type_kw_reclass = {#regex actually here should not be necessary due to standardization
                        'km' : r"\b(kilometer|kilometre|km)s?(?!\s*(\*\*\s*2|\^2|²|square|squared|2))",
                        'km**2' : r"\b(kilometer|kilometre|km)s?\s?(\*\*\s*2|\*\*2|\^2|²|square|squared|2)",
                        'kg' : r"(kg.*|.*kilogram.*)",
                        'm**3' : r"\b(meter|metre|m)s?\s?(\*\*\*\s*3|\*\*3|\^3|³|cube|cubic|3)",
                        '%' : r"(%|perc.*)",
}
unit_kw_reclass ={
                        'people': r"people.*|person.*|women.*|men.*|child.*|children.*|adult.*|adults.*|elder.*|elderly.*|infant.*|infants.*|individual.",
                        'roads' : r"road.*|route.*|bridge.*|highway.*|motorway.*",#r"(?<!kilometer|kilometre|km).*(road.*|route.*|.*bridge.*|.*highway.*|.*motorway.*)",
                        'transportation facilities' : r"rail.*|train track.*|airport.*|\scar.*|railway.*|train.*|bus.*|taxi.*|taxicab.*|truck.*",
                        'water, sanitation and hygiene facilities' : r"water.*|sanitation.*|hygiene.*|latrine.*|well.*|tap.*|reservoir.*|aqueduct.*",
                        'healthcare facilities' : r"health|hospital.*|clinic.*|maternity.*|medical",
                        'IT and communication facilities' : r"communication.*|radio.*|tv.*|cell tower.*|antenna.*",
                        'homes' : r"residential.*|residence.|hous.*|home.*|building.*",
                        'education facilities' : r"education.*|school.*|university.*|college.*",
                        'crop production and forestry' : r"crop.*|field.*|forest.*|tree.*|banana.*|coffee.*|cocoa.*|cotton.*|maize.*|rice.*|sorghum.*|soybean.*|sugar.*|tobacco.*|wheat.*",
                        'agricultural facilities' : r"irrigation.*|barn.*|farm.*",
                        'affected animals' : r"livestock.*|animal.*|fish.*|cow.*|sheep.*|poult.*|cattle.*|goat.*|pig.*|chick.*|horse.*|heads?",
                        'informal settlements' : r"camp.?|tent.?|refuge.?|settlement.?"
                         }
default_subtype_unit = {
 'Affected People': "people",
 'Injured People': "people",
 'Displaced People': "people",
 'Homeless People': "people",
 'Missing People': "people",
 'Human Deaths': "people",
 'Residential Buildings': "homes",
 'Informal settlements': "undefined informal settlements",
 'Education Infrastructure': "schools",
 'Human Health and Wellbeing' : "unknown",
 'Infected and Ill People': "people",
 'Road Infrastructure' : "roads",
 'Other Transportation Infrastructure' : "undefined other transportation infrastructure",
 'Water, Sanitation, and Hygiene Infrastructure': "undefined WASH facilities",
 'Healthcare Infrastructure': "undefined healthcare facilities",
 'IT and Communication Infrastructure': "undefined IT and communication facilities",
 'Residential Buildings': "houses",
 'Informal settlements': "undefined informal settlements",
 'Education Infrastructure': "schools",
 'Power and Energy Production Infrastructure' : "undefined power and energy production infrastructure facilities",
 'Agriculture Infrastructure': "undefined agricultural facilities",
 'Crop Production and Forestry': "undefined crop production and forestry",
 'Affected Livestock and Animals': "undefined affected animals",
 'Other Economic and Livelihood Impacts': "CHF",
 #'Water Quality and Availability':
 'Recreation, Tourism, and Culture' : "unknown",
 'Access to Healthcare': "people",
 'Access to transport and Mobility': "people",
 'Water Quality and Availability' : "unknown",
 'Access to Education':"people",
 'Access to Power and Energy':"people",
 'Access to Food':"people",
 'Access to Water, Sanitation, and Hygiene':"people",
 'Other Human Impacts': "unknown",
 'Other Infrastructure Impacts': "unknown",
 'Other Agricultural Impacts': "unknown",
 'Other Service Access Impacts': "people"
}

def convert_unit(extracted_data, unit_converter):
    """Convert units that can be converted e.g. families => people"""
    def convert(x):
        unit = x['impactUnit']
        if not isinstance(unit, str):
            return x  # skip if unit is None or not a string

        unit = unit.strip()
        if unit == "":
            return x

        for old_unit, (conv_fact, new_unit) in unit_converter.items():
            try:
                if unit == old_unit:
                    x["impactValue"] = float(x["impactValue"]) #force conversion to float
                    x["impactValue"] = conv_fact*x["impactValue"]
                    x["impactUnit"] = new_unit
            except Exception as e:
                print(f"Skipping unit conversion for row due to error: {e}")
                continue
        return x
    extracted_data = extracted_data.apply(convert, axis=1)
    return extracted_data


def assign_unit_type(extracted_data, unit_type_kw_reclass):
    """Detect if dimension of unit can be identified e.g. length, mass,...
       Default to "other"
    """
    def assign_type(x):
        unit = str(x["impactUnit"]).lower() #ensure unit is string
        candidates = [unit_type for unit_type in unit_type_kw_reclass.keys() if re.search(unit_type_kw_reclass[unit_type], unit, re.IGNORECASE)]
        if len(candidates) == 1:
            return candidates[0]
        elif len(candidates) == 0:
            return "other"
        else:
            return "multiple"
    extracted_data["unit_type"] = extracted_data.apply(assign_type, axis=1)
    return extracted_data

def reclassify_units(extracted_data, unit_kw_reclass, default_subtype_unit):
    def reclass_units(x):
        unit = str(x["impactUnit"]).lower() #ensure unit is string
        unit_type = x['unit_type']
        unit_prefix = f"{unit_type} of " if unit_type != "other" else ""
        candidates = [unit_corr for unit_corr in unit_kw_reclass.keys() if re.search(unit_kw_reclass[unit_corr], unit, re.IGNORECASE)]
        if len(candidates) == 1:
            return unit_prefix+candidates[0]
        else:
            #no unit identified, infer unit from category
            inferred_unit = default_subtype_unit[x["impactSubtype"]] if x["impactSubtype"] != "Unknown" else unit
            return unit_prefix+inferred_unit
    extracted_data["impactUnit"] = extracted_data.apply(reclass_units, axis=1)
    return extracted_data

from src.text_processing_functions import *
def standardize_units(value, unit):
    """Standardize units to a common baseline in text"""

    ureg = UnitRegistry()
    #print(f"{value} {unit}")
    identified_units = []
    identified_patterns = []
    for target_unit, unit_patterns in std_unit_kw_reclass.items():
        for pattern in unit_patterns:
            if re.search(pattern, unit, re.IGNORECASE):
                identified_units.append(target_unit)
                identified_patterns.append(pattern)
    #matched = [(target_unit, unit_patterns) for target_unit, unit_patterns in std_unit_kw_reclass.items() if np.any([re.search(pattern, unit, re.IGNORECASE) for pattern in unit_patterns])]
    if len(identified_units) == 0:
        return pd.Series({"impactValue": value, "impactUnit": unit})
    elif len(identified_units) > 1:
        raise ValueError(f"Multiple potential units found for token: {unit}")
    identified_unit = identified_units[0]
    identified_pattern = identified_patterns[0]
    si_unit = unit_mapping[identified_unit]
    # Perform conversion
    quantity = float(value) * ureg(identified_unit)
    converted_quantity = quantity.to(si_unit)
    converted_value = converted_quantity.magnitude
    converted_unit = re.sub(identified_pattern, si_unit, unit)

    return pd.Series({"impactValue": converted_value, "impactUnit": converted_unit})

def standardize_value_units(response_df):
    def join_value_units(x):
        return str(x["impactValue"]) +  "," + str(x["impactUnit"])
    def split_value_units(x):
        return x["value_unit"].split(",")
    def apply_std_units(x):
        return standardize_units(str(x["impactValue"]), str(x["impactUnit"]))
    #response_df["value_unit"] = response_df.apply(join_value_units, axis=1)
    #response_df["value_unit"] = response_df.apply(split_value_units, axis=1)
    response_df[["impactValue", "impactUnit"]]  = response_df.apply(apply_std_units, axis=1)
    return response_df



In [34]:
unit_type = "kg"
unit = "kg of crops"
[unit_corr for unit_corr in unit_kw_reclass.keys() if re.search(unit_kw_reclass[unit_corr], unit, re.IGNORECASE)]


['crop production and forestry']

In [35]:
test_df = pd.DataFrame({
        "impactSubtype": ["Affected People", "Crop Production and Forestry", "Crop Production and Forestry"],
        "impactValue": [1000,1000,100],
        "impactUnit": ["families", "kg of crops","hectares of crops"]})
test_df = standardize_value_units(test_df)
test_df = convert_unit(test_df, unit_converter)
test_df = assign_unit_type(test_df, unit_type_kw_reclass)
test_df = reclassify_units(test_df, unit_kw_reclass, default_subtype_unit)
test_df

,impactSubtype,impactValue,impactUnit,unit_type
0,Affected People,3000.0,people,other
1,Crop Production and Forestry,1000.0,kg of crop production and forestry,kg
2,Crop Production and Forestry,1.0,km**2 of crop production and forestry,km**2


In [36]:
#keep orig unit and value for comparison
response_df_proc["impactValueOrig"] = response_df_proc["impactValue"]
response_df_proc["impactUnitOrig"] = response_df_proc["impactUnit"]
response_df_proc = standardize_value_units(response_df_proc)
response_df_proc = convert_unit(response_df_proc, unit_converter)
response_df_proc = assign_unit_type(response_df_proc, unit_type_kw_reclass)
response_df_proc = reclassify_units(response_df_proc, unit_kw_reclass, default_subtype_unit)

In [37]:
response_df_proc[["impactSubtype","impactValueOrig", "impactValue", "impactUnitOrig", "impactUnit","impactsAnnotation"]]

,impactSubtype,impactValueOrig,impactValue,impactUnitOrig,impactUnit,impactsAnnotation
0,Affected People,43880.0,43880.0,people,people,[TC PAM Global Relief Response : Across5 count...
1,Affected People,34573.0,34573.0,people,people,[TC PAM Global Recovery Response : Across5 cou...
2,Residential Buildings,900.0,900.0,houses,homes,[This is important feedback as the safe shelte...
3,Human Health and Wellbeing,85.0,85.0,per cent,unknown,"[In Vanuatu, the results of feedback mechanism..."
4,"Access to Water, Sanitation, and Hygiene",138.0,414.0,households,people,[A total of13 rainwater harvesting systems ( R...
...,...,...,...,...,...,...
272,Crop Production and Forestry,NaN,nan,NaN,undefined crop production and forestry,"[projected production levels were minimal, and..."
273,Affected Livestock and Animals,NaN,nan,NaN,undefined affected animals,[almost half of surveyed households that kept ...
274,Other Economic and Livelihood Impacts,11.0,11.0,CHF million,CHF,"[the IFRC, in support to the ZRCS, launched an..."
275,Access to Food,NaN,nan,NaN,people,[decreased access to water has also led to out...


In [38]:
# save
savename = "post_processed_" + res_savename
response_df_proc.to_csv(DATA_OUT_LLMS + savename, index=False)